In [7]:
%pip install ortools

Note: you may need to restart the kernel to use updated packages.


In [13]:
import math
import matplotlib.pyplot as plt
import numpy as np
from optimizer.construct_data_objects import SupplyChainData, SimulationParameters
from ortools.linear_solver import pywraplp
from optimizer.math_model_declaration import create_math_model
from optimizer.math_model_constraints import minimize_cost_objective
from ortools_objects.model import ORToolsCPModel
from ortools_objects.var import ScalarORBoolVariable
import pandas as pd
import utils.log as log


logger = log.get_logger("SCIPSolver")

Overview- Model is looking at a classic MFG, Distribution, Store Supply Chain optimization.

Each Manufacturing site has a total capacity, and different cost for shipping to each distribution site.

Each Distribution site receives products from a MFG Site, then fulfilled a Store's replenishment demand, with an associated shipping cost. When a distribution site is opened it must remain opened for x number of days

Each distribution has a fixed recurring cost as well as capacity.

Each store has demand that is must fulfill from the MFG site through the distribution to the final store.

Objective is to minimize the total cost of shipping, plus the cost of maintaining a distribution site.

Model:
Demand is randomly generated for each customer for a 10 day period

Global- looks at the total demand minimizes the cost across the entire horizon
Daily - this model is naive and must make decision on a daily, but it inherits decisions from the previous day. Ex. if a distribution site was opened the day before, it must remain open during this time.

RL - we simulated out the model, and try to do daily decision based on our RF model.

Goal is to compare Global, vs daily vs RL and see how well they converge.


In [9]:
num_days = 10
num_simulations = 10
decision_rolling_period = 3
num_manufacturing_sites = 2
num_distribution_sites = 5
num_customers = 12

distribution_opening_costs = [350, 320, 375, 400, 550]
mfg_site_capacity = [600000, 600000]
mean_demand = [20, 30, 25, 40, 35, 28, 32, 50, 26, 38, 34, 27]
std_dev_demand = [20, 18, 15, 20, 20, 5, 5, 12.4, 12.6, 13.8, 13.4, 12.7]

# Transportation costs
transport_cost_m_to_d = [
    [3.5, 2.5, 4.5, 2.5, 3.0],  # Manufacturing site 1
    [2.5, 4.5, 5.5, 6.5, 8.5]  # Manufacturing site 2
]
transport_cost_d_to_c = [
    [1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 1
    [2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 2
    [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 3
    [2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2],  # Distribution site 4
    [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1]   # Distribution site 5
]

In [10]:
# Create class to organize supply chain data
supply_chain_data = SupplyChainData()
sim_params = SimulationParameters(num_days, num_simulations, decision_rolling_period)

# Adding manufacturing sites. Simply assigning ids based on idx
mf_ids = [mf_id for mf_id in range(num_manufacturing_sites)]
for dist_id in mf_ids:
    supply_chain_data.add_manufacturing_site(site_id=dist_id, capacity=mfg_site_capacity[dist_id])

# Adding distribution sites
dist_ids = [dist_ids for dist_ids in range(num_distribution_sites)]
for dist_id in dist_ids:   
    supply_chain_data.add_distribution_site(site_id=dist_id, opening_cost=distribution_opening_costs[dist_id])

# Adding customers
cust_ids = [cust_id for cust_id in range(num_manufacturing_sites)]
for customer_id in cust_ids:
    supply_chain_data.add_customer(customer_id=customer_id, mean_demand=mean_demand[customer_id], std_dev_demand=std_dev_demand[customer_id])

# Set transport costs for manufacturing sites
for mf_id in range(num_manufacturing_sites):
    for dist_id in range(num_distribution_sites):
        supply_chain_data.manufacturing_sites[mf_id].set_mf_to_dist_transport_costs(transport_cost_m_to_d[mf_id][dist_id])
        
# Set transport costs for distribution sites
for dist_id in range(num_distribution_sites):
    for cust_id in range(num_customers):
        supply_chain_data.distribution_sites[dist_id].set_dist_to_cust_transport_costs(transport_cost_d_to_c[dist_id][cust_id])

In [11]:
or_math_model = ORToolsCPModel(
    logger=logger,
    max_time=30,
    rel_gap=0.00,
    solver_log=True,
    shallow_substitute=True,
)

create_math_model(
    or_math_model,
    supply_chain_data,
    sim_params,
)


In [14]:
or_math_model.construct_model()

status = or_math_model.solve_model()
if status != pywraplp.Solver.OPTIMAL:
    logger.info("Optimizer didn't find optimal solution")
else:
    print(f"total_cost: {minimize_cost_objective(or_math_model)}")
    logger.info("Optimal solution Found")

2024-04-13 07:58:43 - INFO - constraint.py - construct - Began adding constraint distribution_opening_duration to model.


AttributeError: 'ORToolsCPModel' object has no attribute 'decision_rolling_period'

In [17]:

def run_global_solve(all_days_demand, open_duration, opening_costs, transport_cost_m_to_d, transport_cost_d_to_c, supply_capacity, num_days, num_distribution_centers, num_manufacturing_sites, num_customers):
    # Create a new solver instance
    solver = pywraplp.Solver.CreateSolver('SCIP')

    # Global Decision Variables
    transport_m_to_d_global = {}
    transport_d_to_c_global = {}
    open_dc_global = [[solver.BoolVar(f'open_dc_global_{d}_{day}') for day in range(num_days)] for d in range(num_distribution_centers)]


    for d in range(num_distribution_centers):
        for day in range(num_days):
            for m in range(num_manufacturing_sites):
                transport_m_to_d_global[(m, d, day)] = solver.NumVar(0, solver.infinity(), f'transport_m{m}_d{d}_day{day}')

            for c in range(num_customers):
                transport_d_to_c_global[(d, c, day)] = solver.NumVar(0, solver.infinity(), f'transport_d{d}_c{c}_day{day}')

    # Global Objective Function
    total_global_cost = solver.Sum([opening_costs[d] * open_dc_global[d][day]
                                    for d in range(num_distribution_centers)
                                    for day in range(num_days)]) + \
                        solver.Sum([transport_cost_m_to_d[m][d] * transport_m_to_d_global[(m, d, day)]
                                    for m in range(num_manufacturing_sites)
                                    for d in range(num_distribution_centers)
                                    for day in range(num_days)]) + \
                        solver.Sum([transport_cost_d_to_c[d][c] * transport_d_to_c_global[(d, c, day)]
                                    for d in range(num_distribution_centers)
                                    for c in range(num_customers)
                                    for day in range(num_days)])

    solver.Minimize(total_global_cost)


    # Constraints

    # Supply Constraints based on DC status (Big M)
    for d in range(num_distribution_centers):
        for day in range(num_days):
            for c in range(num_customers):
                solver.Add(transport_d_to_c_global[(d, c, day)] <= open_dc_global[d][day] * 100000)
            for m in range(num_manufacturing_sites):
                solver.Add(transport_m_to_d_global[(m, d, day)] <= open_dc_global[d][day] * 100000)


    # Rolling Constraint for distribution center opening duration
    for d in range(num_distribution_centers):
        for day in range(num_days - open_duration + 1):
            for duration_day in range(day + 1, day + open_duration):
                solver.Add(open_dc_global[d][day] <= open_dc_global[d][duration_day])


    # Supply Constraints so total mfg site supply is not more than capacity
    for m in range(num_manufacturing_sites):
        for day in range(num_days):
            total_supply_m = solver.Sum([transport_m_to_d_global[(m, d, day)] for d in range(num_distribution_centers)])
            solver.Add(total_supply_m <= supply_capacity[m])

    # Limit distribution center shipments to the total of received shipments
    for d in range(num_distribution_centers):
        for day in range(num_days):
            total_shipment_from_m_to_d = solver.Sum([transport_m_to_d_global[(m, d, day)] for m in range(num_manufacturing_sites)])
            total_shipment_from_d_to_c = solver.Sum([transport_d_to_c_global[(d, c, day)] for c in range(num_customers)])
            solver.Add(total_shipment_from_m_to_d == total_shipment_from_d_to_c)

    # Demand Constraints: Ensure all shipments to customer meet demand
    for c in range(num_customers):
        for day in range(num_days):
            total_demand_c = solver.Sum([transport_d_to_c_global[(d, c, day)] for d in range(num_distribution_centers)])
            solver.Add(total_demand_c >= all_days_demand[day][c])

    # Solve the problem
    status = solver.Solve()
    if status == pywraplp.Solver.OPTIMAL:
        # Calculate transportation costs
        total_transport_cost_m_to_d = sum(transport_cost_m_to_d[m][d] * transport_m_to_d_global[(m, d, day)].solution_value()
                                          for m in range(num_manufacturing_sites)
                                          for d in range(num_distribution_centers)
                                          for day in range(num_days))

        total_transport_cost_d_to_c = sum(transport_cost_d_to_c[d][c] * transport_d_to_c_global[(d, c, day)].solution_value()
                                          for d in range(num_distribution_centers)
                                          for c in range(num_customers)
                                          for day in range(num_days))


        total_dc_cost = 0
        days_dc_open = [0 for _ in range(num_distribution_centers)]

        for day in range(num_days):
            for distribution in range(num_distribution_centers):
                dc_solution = open_dc_global[distribution][day].solution_value()

                if dc_solution == 1:
                    print(f'Needed {distribution} on day {day}')

                # If the distribution center is opened and has not been opened before
                if dc_solution == 1 and days_dc_open[distribution] == 0:
                    total_dc_cost += opening_costs[distribution]
                    days_dc_open[distribution] += 1
                    print(f'Needed {distribution} on day {day} {opening_costs[distribution]}')

                # If the distribution center is closed but has been open previously within the rolling window
                if dc_solution == 0 and  days_dc_open[distribution] < open_duration and days_dc_open[distribution] >0:
                    days_dc_open[distribution] += 1

                if dc_solution == 1 :
                  print(f"Not Needed {distribution} on day {day} '{days_dc_open[distribution]}")


                # If the distribution center has been open for the entire duration, reset the count
                if days_dc_open[distribution] >= open_duration - 1:
                    days_dc_open[distribution] = 0
                    print(f'Reset {distribution} on day {day}')
                    
        print(f'global cost dc: {total_dc_cost}')
        total_transport_cost = total_transport_cost_m_to_d + total_transport_cost_d_to_c + total_dc_cost

        # Return objective value, DC opening decision, and total transportation cost
        return [[open_dc_global[d][day].solution_value() for day in range(num_days)] for d in range(num_distribution_centers)], total_transport_cost

    else:
        return [0 for _ in range(num_distribution_centers) for _ in range(num_days)],float('inf'),


In [18]:
def run_daily_solve(simulated_demand, open_dc_state, opening_costs, transport_cost_m_to_d, transport_cost_d_to_c, supply_capacity, num_distribution_sites, num_manufacturing_sites, num_customers):
    
    solver = pywraplp.Solver.CreateSolver('SCIP')

    transport_m_to_d_global = {}
    transport_d_to_c_global = {}
    open_dc_decisions = {}

    adj_opening_cost = [0 if state == 1 else cost for state, cost in zip(open_dc_state, opening_costs)]

    for d in range(num_distribution_sites):
        open_dc_decisions[d] = solver.BoolVar(f'open_dc_decision{d}')
        solver.Add(open_dc_decisions[d] >= open_dc_state[d])

    # Global Objective Function
    total_global_cost = solver.Sum([adj_opening_cost[d] * open_dc_decisions[d]
                                    for d in range(num_distribution_sites)])

    ### Decisions ###
    # Shipping Decisions
    for m in range(num_manufacturing_sites):
        for d in range(num_distribution_sites):
            transport_m_to_d_global[(m, d)] = solver.NumVar(0, solver.infinity(), f'transport_m{m}_d{d}')
            total_global_cost += transport_cost_m_to_d[m][d] * transport_m_to_d_global[(m, d)]


    # dist to customer
    for d in range(num_distribution_sites):
        for c in range(num_customers):
            transport_d_to_c_global[(d, c)] = solver.NumVar(0, solver.infinity(), f'transport_d{d}_c{c}')
            total_global_cost += transport_cost_d_to_c[d][c] * transport_d_to_c_global[(d, c)]


    solver.Minimize(total_global_cost)


  # Constraints
    # Supply Constraints based on DC status (Big M)
    for d in range(num_distribution_sites):
            for c in range(num_customers):
                solver.Add(transport_d_to_c_global[(d, c)] <= open_dc_decisions[d] * 100000)
            for m in range(num_manufacturing_sites):
                solver.Add(transport_m_to_d_global[(m, d)] <= open_dc_decisions[d] * 100000)


    # Supply Constraints so total mfg site supply greater than shipped
    for m in range(num_manufacturing_sites):
            total_supply_m = solver.Sum([transport_m_to_d_global[(m, d)] for d in range(num_distribution_sites)])
            solver.Add(total_supply_m <= supply_capacity[m])


    # limit distibution shipments to be equal to the total of recieved shipments
    for d in range(num_distribution_sites):
            total_shipment_from_m_to_d = solver.Sum([transport_m_to_d_global[(m, d)] for m in range(num_manufacturing_sites)])
            total_shipment_from_d_to_c = solver.Sum([transport_d_to_c_global[(d, c)] for c in range(num_customers)])
            solver.Add(total_shipment_from_m_to_d == total_shipment_from_d_to_c)

    # Demand Constraints sum all shipments into customer and make sure it meets demand
    for c in range(num_customers):
            total_demand_c = solver.Sum([transport_d_to_c_global[(d, c)] for d in range(num_distribution_sites)])
            solver.Add(total_demand_c >= simulated_demand[c])


    # Solve the global problem
    status = solver.Solve()

    if status == pywraplp.Solver.OPTIMAL:
        binary_decisions = [open_dc_decisions[d].solution_value() for d in range(num_distribution_sites)]
        # print( binary_decisions)
        return solver.Objective().Value(), binary_decisions
    else:
        print('Solve Not Optimal')
        return float('inf'), [0 for _ in range(num_distribution_sites)]

####



In [19]:
num_manufacturing_sites = 2
num_distribution_sites = 5
num_customers = 12
num_days = 10
num_simulations = 10
decision_rolling_period = 3

distribution_opening_costs = [350, 320, 375, 400, 550]
# mfg_site_capacity = [600000, 600000]

mean_demand = [20, 30, 25, 40, 35, 28, 32, 50, 26, 38, 34, 27]
std_dev_demand = [20, 18, 15, 20, 20, 5, 5, 12.4, 12.6, 13.8, 13.4, 12.7]

#mean_demand = [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
#std_dev_demand = [0,0,0,0,0,0,0,0,0,0,0,0]

# Transportation costs
transport_cost_m_to_d = [
    [3.5, 2.5, 4.5, 2.5, 3.0],
    [2.5, 4.5, 5.5, 6.5, 8.5]
]
transport_cost_d_to_c = [
    [1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 1
    [2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 2
    [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2],  # Distribution site 3
    [2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2],  # Distribution site 4
    [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1]   # Distribution site 5
]



In [20]:

average_dc_open = [0 for _ in range(num_distribution_sites)]
average_difference = []
all_days_demand = []

for simulation in range(num_simulations):

    # Reset tracking variables for each simulation
    open_dc_states = [[0 for day in range(num_days)] for d in range(num_distribution_sites)]
    open_dc_state = [0 for _ in range(num_distribution_sites)]
    days_dc_open = [0 for _ in range(num_distribution_sites)]

    daily_demand_data = []
    total_daily_cost = 0


    for day in range(num_days):
        # Simulate demand for the day for each customer
        simulated_demand = [max(0, np.random.normal(mean_demand[c], std_dev_demand[c])) for c in range(num_customers)]
        daily_demand_data.append(simulated_demand)

        # print(day)
        # Run daily simulation
        daily_cost, distribution_desc = run_daily_solve(simulated_demand, open_dc_state, distribution_opening_costs, transport_cost_m_to_d, transport_cost_d_to_c, mfg_site_capacity, num_distribution_sites, num_manufacturing_sites, num_customers)
        total_daily_cost += daily_cost

        for d in range(num_distribution_sites):
            open_dc_states[d][day] = distribution_desc[d]
            open_dc_state[d] = distribution_desc[d]


        # print(open_dc_state)
        for distribution in range(num_distribution_sites):
            average_dc_open[distribution] += open_dc_state[distribution]

            if open_dc_state[distribution] == 1:
                days_dc_open[distribution] += 1
            else:
                days_dc_open[distribution] = 0

            if days_dc_open[distribution] >= decision_rolling_period:
                open_dc_state[distribution] = 0
                days_dc_open[distribution] = 0



    #print(open_dc_states)
    # Global simulation using the first simulation's demand data
    global_dc_open, global_cost = run_global_solve(daily_demand_data, decision_rolling_period, distribution_opening_costs, transport_cost_m_to_d, transport_cost_d_to_c, mfg_site_capacity, num_days, num_distribution_sites, num_manufacturing_sites, num_customers)
    difference = global_cost-total_daily_cost
    print(f'Look Back Solve Total:{round(global_cost,2)}, Daily Solve Total:{round(total_daily_cost,2)}, Difference: {round(total_daily_cost- global_cost,2)}')
    average_difference.append(difference)

# Calculate average decisions for daily simulations
average_dc_open = [count / (num_simulations * num_days) for count in average_dc_open]

print("Final DC States for Each Decision Point:")
for d in range(num_distribution_sites):
    print(f"Daily Decision Point Dist {d}: {open_dc_states[d]}")
    print(f"Global Decision Point Dist {d}: {global_dc_open[d]}")

print(f"Avg dc open on daily simulation: {average_dc_open}")



Needed 1 on day 0
Needed 1 on day 0 320
Not Needed 1 on day 0 '1
Needed 1 on day 1
Not Needed 1 on day 1 '1
Needed 1 on day 2
Not Needed 1 on day 2 '1
Needed 1 on day 3
Not Needed 1 on day 3 '1
Needed 1 on day 4
Not Needed 1 on day 4 '1
Needed 1 on day 5
Not Needed 1 on day 5 '1
Needed 1 on day 6
Not Needed 1 on day 6 '1
Needed 1 on day 7
Not Needed 1 on day 7 '1
Needed 1 on day 8
Not Needed 1 on day 8 '1
Needed 1 on day 9
Not Needed 1 on day 9 '1
global cost dc: 320
Look Back Solve Total:17035.48, Daily Solve Total:17766.08, Difference: 730.6
Needed 1 on day 0
Needed 1 on day 0 320
Not Needed 1 on day 0 '1
Needed 1 on day 1
Not Needed 1 on day 1 '1
Needed 1 on day 2
Not Needed 1 on day 2 '1
Needed 1 on day 3
Not Needed 1 on day 3 '1
Needed 1 on day 4
Not Needed 1 on day 4 '1
Needed 1 on day 5
Not Needed 1 on day 5 '1
Needed 1 on day 6
Not Needed 1 on day 6 '1
Needed 1 on day 7
Not Needed 1 on day 7 '1
Needed 1 on day 8
Not Needed 1 on day 8 '1
Needed 1 on day 9
Not Needed 1 on day 9 '

In [ ]:
# Output average decisions from daily simulations
print('Average Distribution Site Open Decision over 1000 Simulations and 10 Days:')
for d in range(num_distribution_sites):
    print(f'Distribution Site {d}: {average_dc_open[d]:.2f}')


total_sum = sum(average_difference)
num_elements = len(average_difference)
average = total_sum / num_elements if num_elements > 0 else 0


In [ ]:
# Output results from global simulation
print('\nGlobal Simulation Results:')
print(f'Average Difference:{average}')
print('Open Distribution Sites:', global_dc_open)


Global Simulation Results:
Average Difference:0.0
Open Distribution Centers: [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]


RL Model

In [ ]:


# Define RL parameters
learning_rate = 0.1
discount_factor = 0.9
epsilon = 0.1
num_episodes = 1000

# Initialize Q-table
num_actions = 2  # Open or close distribution site
num_states = num_days * (num_distribution_sites + num_manufacturing_sites + num_customers) * 2  # Number of possible states (open/closed for each distribution site on each day)
q_table = np.zeros((num_states, num_actions))

# Initialize the array for manufacturing site capacity
mfg_site_capacity_array = np.array(mfg_site_capacity)[:, np.newaxis]

# Helper function to convert state tuple to index
def state_to_index(state):
    day, status = state
    distribution_status, manufacturing_status, customer_demand = np.split(status, [num_distribution_sites, num_distribution_sites + num_manufacturing_sites])
    index = day * num_states
    index += np.sum(distribution_status * 2)
    index += np.sum(manufacturing_status * 2) * num_distribution_sites
    index += np.sum(customer_demand) * num_distribution_sites * num_manufacturing_sites
    return index

# RL training
for episode in range(num_episodes):
    total_cost = 0

    # Initialize state
    state = (0, np.zeros(num_states, dtype=int))

    for day in range(num_days):
        # Choose action (epsilon-greedy policy)
        if np.random.rand() < epsilon:
            action = np.random.randint(num_actions)
        else:
            index = state_to_index(state)
            action = np.argmax(q_table[index])

        # Update state
        next_state = (day + 1, state[1].copy())
        next_state[1][action] = 1 - next_state[1][action]

        total_dc_cost = 0
        for distribution, dc_solution in enumerate(next_state[1][:num_distribution_sites]):
            if dc_solution == 1:
                if days_dc_open[distribution] == 0:
                    total_dc_cost += distribution_opening_costs[distribution]
                days_dc_open[distribution] += 1
                if days_dc_open[distribution] >= decision_rolling_period:
                    days_dc_open[distribution] = 0
        total_cost += total_dc_cost

        # Calculate transportation costs from manufacturing sites to distribution sites based on their status
        manufacturing_to_distribution_costs = np.sum(transport_cost_m_to_d * mfg_site_capacity_array * next_state[1][num_distribution_sites:num_distribution_sites+num_manufacturing_sites], axis=0)

        # Calculate transportation costs from distribution sites to customers based on their demand and status
        distribution_to_customer_costs = np.sum(transport_cost_d_to_c * np.dot(next_state[1][:num_distribution_sites][:, np.newaxis], mean_demand[np.newaxis, :]), axis=(0, 1))

        # Add transportation costs to the total cost
        total_cost += np.sum(manufacturing_to_distribution_costs) + distribution_to_customer_costs

        # Update Q-table
        index = state_to_index(state)
        next_index = state_to_index(next_state)
        best_next_action = np.argmax(q_table[next_index])
        q_table[index][action] += learning_rate * (total_cost + discount_factor * q_table[next_index][best_next_action] - q_table[index][action])

        # Move to the next state
        state = next_state

print("Training complete.")

print("Training complete.")

# Evaluate RL agent
# Assuming the RL agent will choose actions greedily
total_costs = []
for episode in range(num_episodes):
    state = (0, np.zeros(num_states, dtype=int))
    total_cost = 0
    for day in range(num_days):
        index = state_to_index(state)
        action = np.argmax(q_table[index])
        total_dc_cost = 0
        for distribution, dc_solution in enumerate(state[1]):
            if dc_solution == 1:
                if days_dc_open[distribution] == 0:
                    total_dc_cost += distribution_opening_costs[distribution]
                days_dc_open[distribution]


NameError: name 'num_days' is not defined